# Splines: Regression Splines and Smoothing Splines

**DSAI 621 — Weeks 4–5.** There are two ways to fit a smooth curve with splines. A **regression spline** chooses a
few knots and fits by least squares. A **smoothing spline** puts a knot at every $x_i$ and controls
flexibility with a penalty $\lambda$ instead. This lab is short. It practices what you need to be
able to do by hand: read a fit off its basis functions, count knots and degrees of freedom, say
where a fit puts its variance, and choose the flexibility by leave-one-out cross-validation.

All of the code is written for you: run each cell, read what it does, and change it to try
things out. Short exercises marked **E1**, **E2**, … ask you to stop and look at what you just
made, and answer in words.
Exercises marked **T1**, **T2**, … ask you to change a number in a cell, rerun it, and
explain what changed. Each one says which cells to rerun and whether to change the number back
afterwards.

**Notation (ISL 7.3–7.5).** A regression spline is $y_i = \beta_0 + \beta_1 b_1(x_i) + \dots +
\beta_J b_J(x_i) + \varepsilon_i$, with $J$ basis functions and a separate intercept. Degrees of
freedom count every coefficient, $\mathrm{df} = J + 1 = \operatorname{tr}(\mathbf{H})$. The truncated
power function is $h(x, \xi) = (x - \xi)_+^3$, which is $(x - \xi)^3$ for $x > \xi$ and 0 otherwise
(ISL 7.10). The smoothing spline $\hat g_\lambda$ minimizes
$\sum_{i=1}^n (y_i - g(x_i))^2 + \lambda \int g''(t)^2\,dt$ (ISL 7.11). Its fitted values are
$\hat{\mathbf{g}}_\lambda = \mathbf{S}_\lambda \mathbf{y}$, and
$\mathrm{df}_\lambda = \operatorname{tr}(\mathbf{S}_\lambda)$ (ISL 7.12–7.13).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.polynomial import Polynomial
from scipy.interpolate import make_smoothing_spline
from scipy.optimize import brentq

# random number generator for reproducibility
rng = np.random.default_rng(621)

### Helper functions

You'll use these throughout; read them, but you don't need to change them.

- `f_true(x)` is the test function from ESL (5.22), $f(x) = \sin(12(x + 0.2))/(x + 0.2)$, and
  `make_data(n, rng, sigma=1)` draws $x_i \sim U[0, 1]$ and $y_i = f(x_i) + \varepsilon_i$.
- `fit_basis(B, y)` adds a column of ones to the basis matrix `B` and returns the least squares
  coefficients. `predict_basis(B, beta)` evaluates the fit.
- `pointwise_se(B, B_new, sigma)` returns the standard error of $\hat f(x_0)$ at each row of `B_new`,
  $\sigma\sqrt{b(x_0)^T(\mathbf{B}^T\mathbf{B})^{-1}b(x_0)}$, and `leverages(B)` returns the diagonal
  $h_{ii}$ of the hat matrix $\mathbf{H}$.
- `h(x, xi, power)` is the truncated power function $(x - \xi)_+^{\text{power}}$; power 0 gives the
  indicator $I(x > \xi)$.
- `poly_basis(x, d)` returns $x, x^2, \dots, x^d$. `trunc_power_basis(x, knots)` returns the cubic
  spline basis $x, x^2, x^3, h(x, \xi_1), \dots, h(x, \xi_K)$ (ISL 7.4.3).
  `natural_spline_basis(x, knots)` returns the natural cubic spline basis of ESL (5.4)–(5.5); its
  `knots` include the two boundary knots. None of them includes the constant: `fit_basis` adds it.

In [ ]:
# true function for this lab
def f_true(x):
    "ESL (5.22): the test function for this lab."
    return np.sin(12 * (x + 0.2)) / (x + 0.2)

# Generate data for this lab
def make_data(n, rng, sigma=1.0):
    "x ~ U[0, 1], y = f(x) + N(0, sigma^2), sorted by x for plotting."
    x = np.sort(rng.uniform(0, 1, n))
    return x, f_true(x) + rng.normal(0, sigma, n)

# Add column of ones to a basis matrix B for the intercept
def with_ones(B):
    return np.column_stack([np.ones(len(B)), B])

# Using lstsq to fit the basis matrix B to the response y, returning the coefficients (including intercept)
# All we need are coefficients, so we don't need to use statsmodels or sklearn for this. 
# The intercept is included in the first column of the basis matrix B.
def fit_basis(B, y):
    "Least squares coefficients (intercept first) for the basis matrix B."
    return np.linalg.lstsq(with_ones(B), y, rcond=None)[0]

# Predict the response at new points B_new using the fitted coefficients beta
def predict_basis(B, beta):
    return with_ones(B) @ beta

# Standard error of fitten values at new points B_new
# Use true sigma, or provide an estimate of sigma from the residuals of the training data.
def pointwise_se(B, B_new, sigma=1.0):
    "SE of f-hat at each row of B_new; B is the training basis."
    G = np.linalg.inv(with_ones(B).T @ with_ones(B))
    Bn = with_ones(B_new)
    return sigma * np.sqrt(np.sum((Bn @ G) * Bn, axis=1))

# Leverages (diagonal of the hat matrix) for the basis B plus an intercept
# Note that the hat matrix is H = B(B^T B)^{-1} B^T, and the diagonal elements h_ii are the leverages.
# Here, we compute the leverages using the pointwise standard errors of the fitted values at the training points B since h_ii = se(fitted value at B_i)^2 / sigma^2.
def leverages(B):
    "Diagonal h_ii of the hat matrix for the basis B plus an intercept."
    return pointwise_se(B, B) ** 2

# Basis functions for polynomials and splines
def poly_basis(x, d):
    "Columns x, x^2, ..., x^d."
    return np.column_stack([x ** j for j in range(1, d + 1)])

# Truncated power function for splines
def h(x, xi, power=3):
    "Truncated power function (x - xi)_+^power (ISL 7.10). power = 0 gives the indicator I(x > xi)."
    return (x > xi).astype(float) if power == 0 else np.clip(x - xi, 0, None) ** power

# Truncated power basis for cubic splines (without the constant)
def trunc_power_basis(x, knots):
    "Cubic spline basis x, x^2, x^3, h(x, xi_1), ..., h(x, xi_K) (ISL 7.4.3), without the constant."
    return np.column_stack([x, x ** 2, x ** 3] + [h(x, k) for k in knots])

# Natural cubic spline basis (without the constant)
def natural_spline_basis(x, knots):
    "Natural cubic spline basis of ESL (5.4)-(5.5), without the constant. knots includes both boundary knots."
    k = np.asarray(knots)
    d = lambda j: (h(x, k[j]) - h(x, k[-1])) / (k[-1] - k[j])
    return np.column_stack([x] + [d(j) - d(len(k) - 2) for j in range(len(k) - 2)])

---
## 1. A fit is a sum of basis functions

To work out what a fitted spline looks like, take it one interval at a time. Left of a knot $\xi$
every term $h(x, \xi)$ is 0. Right of it, $h(x, \xi) = (x - \xi)^3$ is an ordinary cubic that you
can expand. The cell below fits nothing. It takes the cubic spline with one knot at $\xi = 1$,

$$\hat f(x) = 1 - x + 0.5x^2 + 2\,h(x, 1),$$

recovers the cubic on each side of the knot, and evaluates $\hat f$ and its first three derivatives
at the knot from each side.

In [ ]:
xi = 1.0           # the knot
beta_knot = 2.0    # coefficient on the truncated power term
power = 3          # 3 gives h(x, xi) = (x - xi)_+^3, a cubic spline


def f_hat(x):
    return 1 - x + 0.5 * x ** 2 + beta_knot * h(x, xi, power)


# Each side of the knot is a cubic, so a degree-3 fit to points on one side recovers it exactly
# Polynomial.fit returns a Polynomial object, which has a .coef attribute for the coefficients and a .deriv(m) method for the m-th derivative.
xs = np.linspace(-1, 3, 801)
left = Polynomial.fit(xs[xs < xi], f_hat(xs[xs < xi]), 3).convert()
right = Polynomial.fit(xs[xs > xi], f_hat(xs[xs > xi]), 3).convert()
show = lambda p: ", ".join(f"{name} = {c:.2f}" for name, c in zip("abcd", p.coef.round(10) + 0.0))
print("x < xi:  f-hat = a + b x + c x^2 + d x^3 with", show(left))
print("x > xi:  f-hat = a + b x + c x^2 + d x^3 with", show(right))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(xs, f_hat(xs), "tab:blue", lw=2)
ax.axvline(xi, color="grey", ls="--", lw=0.8, label=f"knot $\\xi$ = {xi}")
ax.set(xlabel="x", ylabel=r"$\hat f(x)$", title=f"power = {power}, coefficient on the knot term = {beta_knot}")
ax.legend()
plt.show()

# Value of f-hat and its derivatives at the knot, approached from each side
names = ["f", "f'", "f''", "f'''"]
pd.DataFrame({"from the left": [left.deriv(m)(xi) for m in range(4)],
              "from the right": [right.deriv(m)(xi) for m in range(4)]}, index=names).assign(
    jump=lambda d: d["from the right"] - d["from the left"]).round(4) + 0.0

**E1.** Which of $\hat f$, $\hat f'$, $\hat f''$ and $\hat f'''$ are continuous at the knot? The
coefficient on $h(x, 1)$ is 2: how is that related to the jump in $\hat f'''$?

<details>
<summary>Show answer: E1</summary>

**E1. Answer.** $\hat f$, $\hat f'$ and $\hat f''$ are continuous: each has jump 0. Only
$\hat f'''$ jumps, from 0 to 12. The extra term $2(x - 1)^3$ and its first two derivatives,
$6(x - 1)^2$ and $12(x - 1)$, are all 0 at $x = 1$, so they change nothing at the knot. Its third
derivative is the constant $2 \cdot 6 = 12$. In general a coefficient $\beta$ on $h(x, \xi)$ makes
$\hat f'''$ jump by $6\beta$ at $\xi$, and that is the only thing the knot can change: each knot
adds exactly one df.

</details>

**E2.** By hand, expand $\hat f(x)$ for $x > 1$ into a single cubic $a + bx + cx^2 + dx^3$, and
check your answer against the second line printed above.

<details>
<summary>Show answer: E2</summary>

**E2. Answer.** $(x - 1)^3 = x^3 - 3x^2 + 3x - 1$, so for $x > 1$
$$\hat f(x) = 1 - x + 0.5x^2 + 2x^3 - 6x^2 + 6x - 2 = -1 + 5x - 5.5x^2 + 2x^3,$$
which matches $a = -1$, $b = 5$, $c = -5.5$, $d = 2$. As a check, both pieces give $\hat f(1) = 0.5$.

</details>

**T1.** Set `power = 0`, then `1`, then `2`, rerunning the cell each time. For each, which is
the lowest derivative that jumps at the knot, and by how much? Describe the shape of the curve at
the knot. Set `power` back to 3 afterwards.

<details>
<summary>Show answer: T1</summary>

**T1. Answer.**

| `power` | term | what jumps | jump | shape at the knot |
|---|---|---|---|---|
| 0 | $2\,I(x > 1)$ | $\hat f$ | $0.5 \to 2.5$, i.e. 2 | a step (a jump in the curve) |
| 1 | $2(x - 1)_+$ | $\hat f'$ | $0 \to 2$, i.e. 2 | continuous, with a kink: the slope changes by 2 |
| 2 | $2(x - 1)_+^2$ | $\hat f''$ | $1 \to 5$, i.e. 4 | smooth to the eye; the curvature changes |
| 3 | $2(x - 1)_+^3$ | $\hat f'''$ | $0 \to 12$, i.e. 12 | smooth; this is the cubic spline |

A term $\beta(x - \xi)_+^r$ leaves $\hat f, \dots, \hat f^{(r-1)}$ continuous and makes
$\hat f^{(r)}$ jump by $r!\,\beta$ (with $r = 0$ the indicator, a jump of $\beta$ in $\hat f$
itself). To sketch a fit built from indicators and truncated lines, write $\hat f$ on each
interval: an indicator adds a step and a truncated line adds a change of slope.

</details>

---
## 2. Counting degrees of freedom, and where the variance goes

A cubic spline with $K$ interior knots has $J = K + 3$ basis functions ($x, x^2, x^3$ and one
$h(x, \xi_k)$ per knot), so $\mathrm{df} = K + 4$. A natural cubic spline must also be linear beyond
its two boundary knots. That costs 2 constraints at each end, so with $K$ interior knots it has
$\mathrm{df} = K + 2$. For the same df, the natural spline gets two more interior knots.

For any basis, $\operatorname{Var}\hat f(x_i) = \sigma^2 h_{ii}$ and
$\sum_i h_{ii} = \operatorname{tr}(\mathbf{H}) = \mathrm{df}$. So the *total* variance over the
training points is $\mathrm{df}\cdot\sigma^2$ and the *average* is $\sigma^2\,\mathrm{df}/n$,
whatever the basis. The basis only decides *where* along $x$ that variance goes. The cell gives a
polynomial, a cubic spline and a natural spline the same df on $n = 50$ equally spaced points, and
plots the standard error of each fit (with $\sigma = 1$; it doesn't depend on $y$).

In [ ]:
dof = 6          # degrees of freedom of every fit, including the intercept
n_design = 50

# Design points for the basis functions, and evaluation points for the standard errors
x_design = (np.arange(n_design) + 0.5) / n_design       # equally spaced in (0, 1)
x0 = np.linspace(0, 1, 201)

# This part creates the basis functions for polynomials and splines
K_cubic, K_natural = dof - 4, dof - 2                     # interior knots for the same df
cs_knots = np.quantile(x_design, np.linspace(0, 1, K_cubic + 2)[1:-1])
ns_knots = np.quantile(x_design, np.linspace(0, 1, K_natural + 2))   # includes the 2 boundary knots
bases = {f"degree-{dof - 1} polynomial": lambda t: poly_basis(t, dof - 1),
         f"cubic spline, K = {K_cubic}": lambda t: trunc_power_basis(t, cs_knots),
         f"natural spline, K = {K_natural}": lambda t: natural_spline_basis(t, ns_knots)}

# For each of the basis functions, we compute the standard errors of the fitted values at the evaluation points x0, and plot them. 
# We also compute some summary statistics about the leverages and standard errors at specific points.
fig, ax = plt.subplots(figsize=(8, 4))
rows = {}
for (name, basis), ls in zip(bases.items(), ["-", "--", "-."]):
    B = basis(x_design)
    se = pointwise_se(B, basis(x0))
    lev = leverages(B)
    ax.plot(x0, se, ls=ls, lw=2, label=name)
    rows[name] = {"J": B.shape[1], "sum of h_ii": lev.sum(), "average Var / sigma^2": lev.mean(),
                  "SE at x0 = 0.5": se[100], "SE at x0 = 1": se[-1]}
ax.set(ylim=(0, 1.2), xlabel="$x_0$", ylabel=r"SE$(\hat f(x_0))\,/\,\sigma$",
       title=f"Same df = {dof}, n = {n_design}: where does the variance go?")
ax.legend()
plt.show()
pd.DataFrame(rows).T.round(3)

**E3.** Check the table: is $J$ the same for all three, and do $K + 4$ and $K + 2$ give the df you
asked for? Is the sum of the $h_{ii}$ the same? Then compare where each basis puts its variance. If
$f$ wiggles a lot in the middle of the data but is close to linear near the ends, which of the
three would you use?

<details>
<summary>Show answer: E3</summary>

**E3. Answer.** All three have $J = 5$ and $\sum_i h_{ii} = 6 = \mathrm{df}$: the cubic spline
with $K = 2$ has $2 + 4 = 6$, and the natural spline with $K = 4$ has $4 + 2 = 6$. So all three
have the same average variance, $6/50 = 0.12\,\sigma^2$. They put it in different places. The
polynomial and the cubic spline have SE 0.265 and 0.258 at $x_0 = 0.5$, but 0.858 and 0.806 at the
edge. The natural spline has a little more in the middle (0.285) and much less at the edge (0.633):
its linear tails cost few df, and it spends the df it saves on two more interior knots. With $f$
wiggly in the middle and nearly linear at the ends, use the natural spline. Its extra knots lower
the bias where $f$ bends, and its linear tails cost little bias where $f$ is nearly linear anyway.

</details>

**T2.** Set `n_design = 200` and rerun. What happens to the sum of the $h_{ii}$, the average
variance and the standard errors? Set it back to 50 afterwards.

<details>
<summary>Show answer: T2</summary>

**T2. Answer.** The sum of the $h_{ii}$ is still 6, because it equals df whatever $n$ is. The
average variance falls from $6/50 = 0.12$ to $6/200 = 0.03$, a quarter. Every SE roughly halves
(for example, the natural spline's edge SE goes from 0.633 to 0.319), but the shape of the curves
and the order of the three bases don't change. With more data each extra df costs less variance,
so cross-validation will choose a larger df.

</details>

---
## 3. Training error, test error and LOOCV for regression splines

Two facts from least squares (DSAI 601) matter here.

- **Nested models.** If you add a knot and *keep* the ones you have, the old fit is still available
  (give the new $h(x, \xi)$ coefficient 0). So the training RSS can't go up.
- **Non-nested models.** If the knots *move* when $K$ changes, as they do when they sit at quantiles
  of $x$, the new model doesn't contain the old one, and the training RSS *can* go up.

Training error is no guide to test error, so we choose df by cross-validation. For a least squares
fit, LOOCV needs only one fit (ISL 5.2):
$\mathrm{CV}_{(n)} = \frac{1}{n}\sum_i \big((y_i - \hat y_i)/(1 - h_{ii})\big)^2$. The cell computes
part (a) for a cubic spline, then fits natural splines with $K = 0, \dots, 15$ interior knots at
quantiles of $x$ and records the training MSE, the LOOCV MSE and the test error
$\sigma^2 + \text{average of }(\hat f - f)^2$ over a fine grid (which we can compute only because we
know $f$).

In [ ]:
x, y = make_data(100, rng)
sigma = 1.0
grid = np.linspace(x.min(), x.max(), 400)

# (a) Nested: add one knot at a time and keep the ones you have where they are
# you should see RSS decrease as you add more knots
added = [0.5, 0.25, 0.75, 0.125, 0.375, 0.625, 0.875]
rss_nested = [np.sum((y - predict_basis(trunc_power_basis(x, added[:K]), fit_basis(trunc_power_basis(x, added[:K]), y))) ** 2)
              for K in range(len(added) + 1)]
print("cubic spline, knots kept and one added each time; training RSS:", np.round(rss_nested, 2))

# (b) Natural spline with K interior knots at quantiles of x: every knot moves when K changes
# you should see RSS can actually increase since the knots are moving around
# we also calculate the LOOCV MSE and the test error for each K, and store them in a DataFrame for comparison
# we can calculate test error since we know the truth
rows = []
for K in range(16):
    knots = np.quantile(x, np.linspace(0, 1, K + 2))
    B = natural_spline_basis(x, knots)
    beta = fit_basis(B, y)
    e = y - predict_basis(B, beta)
    rows.append({"K": K, "df": K + 2,
                 "train MSE": np.mean(e ** 2),
                 "LOOCV MSE": np.mean((e / (1 - leverages(B))) ** 2),          # ISL (5.2)
                 "test error": sigma ** 2 + np.mean((predict_basis(natural_spline_basis(grid, knots), beta) - f_true(grid)) ** 2)})
ns_table = pd.DataFrame(rows).set_index("K")

fig, ax = plt.subplots(figsize=(8, 4))
for col, style in zip(["train MSE", "LOOCV MSE", "test error"], ["o-", "s--", "^-."]):
    ax.plot(ns_table["df"], ns_table[col], style, label=col)
ax.axhline(sigma ** 2, color="grey", ls=":", label=r"$\sigma^2$")
ax.set(xlabel="df", ylabel="mean squared error", title="Natural spline, knots at quantiles of x")
ax.legend()
plt.show()
ns_table.round(3)

**E4.** (a) Does the training RSS in the printed line ever go up? (b) In the table, find
values of $K$ where the training MSE goes *up* when $K$ increases by 1. Why can that happen in (b)
but not in (a)?

<details>
<summary>Show answer: E4</summary>

**E4. Answer.** (a) No: it falls from 263.94 to 92.19 and never rises, because each fit contains
the one before it. (b) For example $K = 1 \to 2$ (2.642 to 2.644), $K = 9 \to 10$ (0.908 to 0.928)
and $K = 12 \to 13$ (0.855 to 0.876). When $K$ changes, all of the quantile knots move, so the model
with $K + 1$ knots doesn't contain the model with $K$ knots. A bigger model that isn't nested can fit
the training data worse.

</details>

**E5.** Which df would you choose using each of the three curves? Which of the three could you
compute on a real data set?

<details>
<summary>Show answer: E5</summary>

**E5. Answer.** The training MSE keeps falling, apart from the bumps in E4, so it would choose
the largest df: it rewards fitting the noise. The test error is smallest at df = 8 ($K = 6$, 1.031).
LOOCV is smallest at df = 10 ($K = 8$, 1.162), but it is nearly flat from df = 6 to 14 (1.16 to
1.24), so it picks the right region. Only the training MSE and LOOCV can be computed on real data:
the test error needs the true $f$. The same $1 - h_{ii}$ shortcut comes back for smoothing splines
in Section 5.

</details>

---
## 4. Smoothing splines

A smoothing spline doesn't choose knots. It minimizes
$$\sum_{i=1}^n \big(y_i - g(x_i)\big)^2 + \lambda\int g''(t)^2\,dt \qquad \text{(ISL 7.11)}$$
over *all* functions $g$. The first term rewards fitting the data; the second penalizes curvature.
For $\lambda > 0$ the minimizer is a **natural cubic spline with a knot at every $x_i$, with its
coefficients shrunk** (ISL 7.5.1). With $n$ coefficients for $n$ points, least squares alone would
interpolate; the penalty is what stops it.

The fit is linear in $\mathbf{y}$: $\hat{\mathbf{g}}_\lambda = \mathbf{S}_\lambda\mathbf{y}$, and
$\mathrm{df}_\lambda = \operatorname{tr}(\mathbf{S}_\lambda) = \sum_i \{\mathbf{S}_\lambda\}_{ii}$
measures its flexibility. scipy's `make_smoothing_spline(x, y, lam)` minimizes exactly (7.11). The
helpers below wrap it, build $\mathbf{S}_\lambda$ one column at a time (column $j$ is the fit to
the $j$-th unit vector, because the fit is linear in $\mathbf{y}$), and find the $\lambda$ that gives
a chosen $\mathrm{df}_\lambda$. scipy keeps using the end cubic piece beyond $x_1$ and $x_n$, so
`smoothing_fit` continues the fit as a straight line there, as a natural spline should. The data are the 100 points from Section 3.

In [ ]:
def smoothing_fit(x, y, lam):
    "The smoothing spline g-hat_lambda of ISL (7.11), returned as a function."
    spl = make_smoothing_spline(x, y, lam=lam)
    slope = spl.derivative()
    lo, hi = x.min(), x.max()

    # scipy's spline keeps using the end cubic piece beyond x_1 and x_n. The smoothing
    # spline is natural, so continue it as a straight line instead.
    def g(t):
        t = np.asarray(t, dtype=float)
        inside = np.clip(t, lo, hi)
        return spl(inside) + slope(inside) * (t - inside)
    return g

# The smoothing matrix S_lambda, which maps y to g-hat_lambda(x)
def smoother_matrix(x, lam):
    "S_lambda. The fit is linear in y, so column j is the fit to the j-th unit vector."
    I = np.eye(len(x))
    return np.column_stack([smoothing_fit(x, I[:, j], lam)(x) for j in range(len(x))])

# The lambda with df_lambda = tr(S_lambda) equal to df (needs 2 < df < n).
# brentq finds a root of a function in a given interval. 
# Here, we are finding the value of lambda such that the trace of the smoothing matrix equals the desired degrees of freedom.
def lam_for_df(x, df):
    "The lambda with df_lambda = tr(S_lambda) equal to df (needs 2 < df < n)."
    return 10 ** brentq(lambda s: np.trace(smoother_matrix(x, 10 ** s)) - df, -14, 6)

The next cell fits the smoothing spline over a wide range of $\lambda$. For each, it reports
$\mathrm{df}_\lambda$, the training RSS, a check that $\mathbf{S}_\lambda\mathbf{y}$ really is the fit,
and how far the fit is from the least squares line. It then prints $\hat g''$ at the two end
points.

In [ ]:
lambdas = [0, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1, 1e2, 1e4]

ls_line = Polynomial.fit(x, y, 1)(x)              # the least squares line
rows = []
for lam in lambdas:
    S = smoother_matrix(x, lam)
    g = smoothing_fit(x, y, lam)
    rows.append({"lambda": lam, "df_lambda = tr(S)": np.trace(S), "training RSS": np.sum((y - g(x)) ** 2),
                 "max |S y - g(x)|": np.abs(S @ y - g(x)).max(),
                 "max |g(x) - LS line|": np.abs(g(x) - ls_line).max()})
display(pd.DataFrame(rows).set_index("lambda").rename(index="{:g}".format).round(4))

curvature = make_smoothing_spline(x, y, lam=1e-4).derivative(2)
print(f"lambda = 1e-4: g''(x_1) = {curvature(x[0]):.1e}, g''(x_n) = {curvature(x[-1]):.1e}")

**E6.** (a) What are $\mathrm{df}_\lambda$ and the training RSS at $\lambda = 0$? What does the fit
do, and what is $\mathbf{S}_0$? (b) As $\lambda$ grows, what does the fit become, and what does
$\mathrm{df}_\lambda$ tend to? Why does it stop at that value and not go to 0? (c) What do the
printed values of $\hat g''$ at the end points tell you?

<details>
<summary>Show answer: E6</summary>

**E6. Answer.** (a) $\mathrm{df}_0 = 100 = n$ and the training RSS is 0. With no penalty the
fit interpolates the data, so $\hat{\mathbf{g}}_0 = \mathbf{y}$ and $\mathbf{S}_0 = I$.
(b) $\mathrm{df}_\lambda$ falls steadily (95.1, 70.7, 33.1, 11.9, 4.53, 2.21, …) toward 2, and the fit
approaches the least squares line (the largest gap is 0.0000 at $\lambda = 10^4$). A straight line
has $g'' = 0$, so its penalty is 0 however large $\lambda$ is: the intercept and slope are never
shrunk, and they always count as 2 df. The training RSS rises from 0 to the least squares line's
266.40. (c) $\hat g'' = 0$ at $x_1$ and $x_n$ (up to rounding). The fit is a *natural* spline: no
data lie beyond the end points, so bending there would add penalty and reduce no residual. (The
zero column shows that $\mathbf{S}_\lambda\mathbf{y}$ is the fit at every $\lambda$.)

</details>

Now plot four smoothing splines, choosing $\lambda$ so that $\mathrm{df}_\lambda$ is 3, 6, 12 and 40.

In [ ]:
target_dfs = [3, 6, 12, 40]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(x, y, s=12, facecolors="none", edgecolors="grey", label="data")
ax.plot(grid, f_true(grid), "k:", lw=2, label="true f")
for df, ls in zip(target_dfs, ["-", "--", "-.", "-"]):
    lam = lam_for_df(x, df)
    ax.plot(grid, smoothing_fit(x, y, lam)(grid), ls=ls, lw=1.5 if df == 40 else 2,
            label=rf"df$_\lambda$ = {df}  ($\lambda$ = {lam:.1e})")
ax.set(ylim=(-4, 5), xlabel="x", ylabel="y", title="Smoothing splines")
ax.legend(loc="upper right", fontsize=9)
plt.show()

**E7.** Which fit looks closest to the true $f$? Describe what goes wrong at df 3 and at df 40.
Look at the $\lambda$ values in the legend: why do we usually report $\mathrm{df}_\lambda$ rather than
$\lambda$?

<details>
<summary>Show answer: E7</summary>

**E7. Answer.** df 12 follows the true $f$ closely. df 3 is almost a straight line and misses the
wave entirely (high bias). df 40 chases the noise, with spikes the true $f$ doesn't have (high
variance). The $\lambda$ values run from about $10^{-1}$ to $4 \times 10^{-7}$, and on their own they
mean nothing: the size of $\lambda$ depends on the units of $x$ (stretching $x$ by a factor $c$
multiplies the right $\lambda$ by $c^3$). $\mathrm{df}_\lambda$ is on the same scale as a regression
spline's df, between 2 and $n$, so it is the number to report and compare.

</details>

---
## 5. Choosing $\lambda$ by LOOCV, in one fit

Leave-one-out cross-validation fits the smoothing spline $n$ times, each time without one
observation, and predicts the point that was left out. For a smoothing spline this needs only *one*
fit (ISL 7.5.2):
$$\mathrm{RSS}_{cv}(\lambda) = \sum_{i=1}^n\big(y_i - \hat g_\lambda^{(-i)}(x_i)\big)^2
= \sum_{i=1}^n\left(\frac{y_i - \hat g_\lambda(x_i)}{1 - \{\mathbf{S}_\lambda\}_{ii}}\right)^2.$$
It is the same formula as for least squares in Section 3, with $\{\mathbf{S}_\lambda\}_{ii}$ in
place of $h_{ii}$. The next cell computes $\mathrm{RSS}_{cv}$ both ways at $\mathrm{df}_\lambda = 8$.

In [ ]:
df_check = 8

lam = lam_for_df(x, df_check)
S = smoother_matrix(x, lam)
e = y - S @ y

# The slow way: refit n times, each time without one observation
loo_pred = np.array([smoothing_fit(np.delete(x, i), np.delete(y, i), lam)(x[i]) for i in range(len(x))])
rss_cv_refit = np.sum((y - loo_pred) ** 2)
# The shortcut: one fit
rss_cv_shortcut = np.sum((e / (1 - np.diag(S))) ** 2)

print(f"df = {np.trace(S):.2f}, lambda = {lam:.2e}")
print(f"RSS_cv by refitting {len(x)} times: {rss_cv_refit:.4f}")
print(f"RSS_cv by the shortcut:         {rss_cv_shortcut:.4f}")
print(f"training RSS:                   {np.sum(e ** 2):.4f}")

**T3.** Set `df_check = 30`, then `99`, rerunning the cell each time. Do the two ways still
agree? Compare the training RSS with $\mathrm{RSS}_{cv}$. What would happen to the shortcut at
$\lambda = 0$? Set `df_check` back to 8 afterwards.

<details>
<summary>Show answer: T3</summary>

**T3. Answer.** They agree every time: 124.7042 both ways at df 30, and 2626.4292 both ways at
df 99. The shortcut is exact, not an approximation. The gap between the training RSS and
$\mathrm{RSS}_{cv}$ grows with df: 98.4 against 120.7 at df 8, 56.5 against 124.7 at df 30, and
0.315 against 2626 at df 99. At df 99 the fit nearly interpolates, so every residual is tiny, but
each $\{\mathbf{S}_\lambda\}_{ii}$ is close to 1 (the smallest $1 - \{\mathbf{S}_\lambda\}_{ii}$ is
about $5 \times 10^{-6}$): each point mostly predicts itself, and dividing by
$1 - \{\mathbf{S}_\lambda\}_{ii}$ undoes that. At $\lambda = 0$, $\mathbf{S}_0 = I$, so every
residual and every $1 - \{\mathbf{S}_\lambda\}_{ii}$ is 0, and the formula is $0/0$.

</details>

Finally, compute the training MSE, the LOOCV MSE ($\mathrm{RSS}_{cv}/n$) and the test error over a
range of $\mathrm{df}_\lambda$, and plot the fit LOOCV chooses.

In [ ]:
df_grid = [2.5, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 30, 50, 80]

rows = []
for df in df_grid:
    lam = lam_for_df(x, df)
    S = smoother_matrix(x, lam)
    e = y - S @ y
    g = smoothing_fit(x, y, lam)
    rows.append({"df": df, "lambda": lam, "train MSE": np.mean(e ** 2),
                 "LOOCV MSE": np.mean((e / (1 - np.diag(S))) ** 2),
                 "test error": sigma ** 2 + np.mean((g(grid) - f_true(grid)) ** 2)})
ss_table = pd.DataFrame(rows).set_index("df")
best_df = ss_table["LOOCV MSE"].idxmin()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col, style in zip(["train MSE", "LOOCV MSE", "test error"], ["o-", "s--", "^-."]):
    axes[0].plot(ss_table.index, ss_table[col], style, label=col)
axes[0].axhline(sigma ** 2, color="grey", ls=":", label=r"$\sigma^2$")
axes[0].set(xscale="log", xlabel=r"df$_\lambda$ (log scale)", ylabel="mean squared error", title="Choosing $\\lambda$")
axes[0].set_xticks(df_grid[::2], [f"{d:g}" for d in df_grid[::2]])
axes[0].minorticks_off()
axes[0].legend()
axes[1].scatter(x, y, s=12, facecolors="none", edgecolors="grey")
axes[1].plot(grid, f_true(grid), "k:", lw=2, label="true f")
axes[1].plot(grid, smoothing_fit(x, y, ss_table.loc[best_df, "lambda"])(grid), "tab:red", lw=2,
             label=rf"LOOCV choice, df$_\lambda$ = {best_df:g}")
axes[1].set(xlabel="x", ylabel="y")
axes[1].legend()
plt.show()
ss_table.assign(**{"lambda": ss_table["lambda"].map("{:.1e}".format)}).round(4)

**E8.** Which $\mathrm{df}_\lambda$ does LOOCV choose, and which minimizes the test error? What
does the training MSE do? How does the best smoothing spline compare with the best natural spline
from Section 3?

<details>
<summary>Show answer: E8</summary>

**E8. Answer.** LOOCV chooses $\mathrm{df}_\lambda = 15$ (1.1585), but the curve is almost flat
from 10 to 20 (1.158 to 1.168), so df 12 (1.1591) is just as good. The test error is smallest at
df 9 (1.076) and is still within 0.1 of that at df 15. The training MSE falls the whole way, to
0.146 at df 80: it always prefers the most flexible fit. The best smoothing spline's test error
(1.076) is about the same as the best natural spline's (1.031 in Section 3). The difference is in
how you choose the flexibility: the regression spline chooses the number and position of its
knots, while the smoothing spline keeps every knot and chooses $\lambda$, which can give any
$\mathrm{df}_\lambda$, not just whole numbers.

</details>

---
## 6. Wrap-up questions

**Q1.** Suppose the penalty were $\lambda\int g'(t)^2\,dt$ (the first derivative) instead. As
$\lambda \to \infty$, what does the fit become, and what does $\mathrm{df}_\lambda$ tend to? What if
the penalty were on $g'''$?

<details>
<summary>Show answer: Q1</summary>

**Q1. Answer.** As $\lambda \to \infty$ the fit must have zero penalty, so with a penalty on
$g'$ it must have $g' = 0$: a constant. The best constant is $\bar y$, and $\mathrm{df}_\lambda \to 1$.
With a penalty on $g'''$ it must have $g''' = 0$: a quadratic. The fit becomes the least squares
quadratic, and $\mathrm{df}_\lambda \to 3$. In general, a penalty on the $m$th derivative gives the
least squares polynomial of degree $m - 1$ as $\lambda \to \infty$; $m = 2$ gives the line from E6.

</details>

**Q2.** "A smoothing spline has $n$ knots, so it has $n$ parameters and must overfit." What is
wrong with this? What *would* you get by fitting a natural cubic spline with a knot at every $x_i$
by least squares, with no penalty?

<details>
<summary>Show answer: Q2</summary>

**Q2. Answer.** The $n$ coefficients are shrunk by the penalty, so they aren't free.
Flexibility is measured by $\mathrm{df}_\lambda = \operatorname{tr}(\mathbf{S}_\lambda)$, and
$\lambda$ sets it anywhere between 2 and $n$ (E6), so it is chosen by LOOCV like any other tuning
parameter. Least squares with no penalty is the $\lambda = 0$ end: $n$ coefficients for $n$ points,
a square basis matrix, $\mathbf{H} = I$, and a curve through every point (training RSS 0).

</details>